# Lab session 3: Federated Learning

**What is federated learning?**
Federated Learning (FL) is a way to train ML models without moving (potentially sensitive) data from one server (e.g. hospital) to another.

<br>Instead of sending all data to one central place, each site (or client):

1. Receives a copy of a global model

2. Trains it locally on its own data

3. Sends **only the updated weights** back

4. A central server averages these updates

5. The averaged global model is redistributed

<br>By doing this:
- No raw data is ever shared, each site can keep its own dataset private (this is important in a healthcare setting)
- All sites still benefit from each other’s data and overall model performance can improve

<p align="center">
  <img src="https://blog.ml.cmu.edu/wp-content/uploads/2019/11/Screen-Shot-2019-11-12-at-10.42.34-AM-1024x506.png"
       width="600">
</p>


## 0. Import and dataset

In [ ]:
# !pip install tensorflow   # only needed if running locally (preinstalled on Colab)

In [ ]:
import os, zipfile, glob
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")

# ensure reproducibility
SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

Get the dataset (**modify this according to your setup**)

In [ ]:
zip_path = "wbc_cr_healthy_medium.zip"
extract_dir = "data/"

# create folder for data dir
os.makedirs(extract_dir, exist_ok=True)

# unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

#data_dir = "/content/data/wbc_cr_healthy_medium/"
data_dir = "data/wbc_cr_healthy_medium/"

What classes are there?
<br>**Hint:** loop through the folders in the dataset

In [ ]:
# YOUR CODE HERE...

Organise dataset as a dataframe (nothing to do here)

In [ ]:
image_paths = []
labels = []
for cls in classes:
    for f in glob.glob(os.path.join(data_dir, cls, '*.png')):
        image_paths.append(f)
        labels.append(cls)
df = pd.DataFrame({'filename': image_paths, 'label': labels})
df.head()

## 1. Exploratory data analysis
- What does the data look like?
- How many classes are there? How many samples per class?
- Are there any data imbalances?

In [ ]:
# YOUR CODE HERE...

Plot some datapoints

In [ ]:
plt.figure(figsize=(15, 9))

samples_per_class = 3

for i, c in enumerate(classes):
    class_path = os.path.join(data_dir, c)
    sample_images = random.sample(os.listdir(class_path), k=samples_per_class)  # 3 random images per class

    for j, img_name in enumerate(sample_images):
        img_path = os.path.join(class_path, img_name)
        img = mpimg.imread(img_path)

        plt.subplot(samples_per_class, len(classes), j*len(classes) + i + 1)
        plt.imshow(img)
        if j == 0:
            plt.title(c, fontsize=12)
        plt.axis('off')

# YOUR CODE HERE...

## 2. Prepare the dataset

To simulate a federated learning scenario, we will create two training centres that have different distributions of the data:


**Centre 1**

- All classes except Monocyte
(or very few Monocyte samples)

**Centre 2**

- All classes except Neutrophile Segment
(or very few Neutrophile Segment samples)

<br>**We therefore split the entire dataset into:**

1. Training set

- This includes the images that each centre will use to train its local model.
- Includes a validation set (not test).

2. Test set (held out until the very end)

- This is a **completely separate** dataset.
- You should not look at it or use it during training.
- Only used once, at the end, to evaluate how well each model generalises.



Create a general train and test split

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
len(df_train), len(df_test)

Allocate data to Centre 1 and Centre 2

<br>**Note:** Even though each centre is supposed to “miss” one class, we still include a very small number of samples from that class. This keeps the setup stable for this illustrative federated learning scenario.
<br>**Hint:** See how changing the number of samples (e.g., 0 samples) affects FL later on.

In [ ]:
print("\nCentre 1")
# Centre 1 lacks Monocyte
df_other_c1 = df_train[df_train.label != "Monocyte"]
# add very few monocytes
df_mono_few = df_train[df_train.label == "Monocyte"].sample(10, random_state=SEED)
df_c1 = pd.concat([df_other_c1, df_mono_few])

print(df_c1.label.value_counts())

print("\nCentre 2")
# Centre 2 lacks NS
df_other_c2 = df_train[df_train.label != "Neutrophile Segment"]
# add very few NS
df_neutro_few = df_train[df_train.label == "Neutrophile Segment"].sample(10, random_state=SEED)
df_c2 = pd.concat([df_other_c2, df_neutro_few])

print(df_c2.label.value_counts())


Split Centre data into training and validation

In [ ]:
df_c1_train, df_c1_val = train_test_split(
    df_c1, test_size=0.1, stratify=df_c1["label"], random_state=SEED)

df_c2_train, df_c2_val = train_test_split(
    df_c2, test_size=0.1, stratify=df_c2["label"], random_state=SEED)

Prepare image generators for CNN classifier
- Centre 1: training and validation
- Centre 2: training and validation
- Global test set

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
datagen = ImageDataGenerator(rescale=1./255)


CLASSES = sorted(os.listdir(data_dir))

print("Centre 1")
train_c1 = datagen.flow_from_dataframe(
    df_c1_train,
    x_col="filename",
    y_col="label",
    classes=CLASSES,       # force full class list to have 5 outputs (important for later)
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    seed=SEED
)
val_c1 = datagen.flow_from_dataframe(
    df_c1_val,
    x_col="filename",
    y_col="label",
    classes=CLASSES,
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    seed=SEED
)

print("\nCentre 2")
train_c2 = datagen.flow_from_dataframe(
    df_c2_train,
    x_col="filename",
    y_col="label",
    classes=CLASSES,
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    seed=SEED
)
val_c2 = datagen.flow_from_dataframe(
    df_c2_val,
    x_col="filename",
    y_col="label",
    classes=CLASSES,
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    seed=SEED
)

print("\nGobal test set")
test_gen = datagen.flow_from_dataframe(
    df_test,
    x_col="filename",
    y_col="label",
    classes=CLASSES,
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED
)


## 3. Train a CNN classifier independently for each centre

Build a simple CNN classifier architecture to be used for all models (Centre 1, Centre 2 and FL model).

<br> **Hint:** Be mindful of the number of parameters so that training time remains reasonable for the lab - you will train quite a few models!

In [ ]:
def build_model():
    model = # YOUR CODE HERE...
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

**Train Centre 1 model**

In [ ]:
model_c1 = # YOUR CODE HERE...
history_c1 = model_c1.fit() # YOUR CODE HERE... don't forget the validation split!

**Train Centre 2 model**

In [ ]:
model_c2 = # YOUR CODE HERE...
history_c2 = model_c2.fit() # YOUR CODE HERE... don't forget the validation split!

**Evaluate** on the whole, **unseen** test set (with all classes)
- Compute accuracy both on *all* classes and on the "seen" classes only.
- Each centre is "missing" one class in its training data, so excluding that class gives a fair measure of how well the model performs on the classes it *actually* saw during training, compared to its overall (global) performance.

In [ ]:
def evaluate(model, generator, class_names, seen_classes, center_name="Model"):

    y_true = np.array(generator.classes).reshape(-1)

    # predict probas
    y_prob = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    # results dict
    results = {}

    # global accuracy
    global_acc = # YOUR CODE HERE...
    results["global_accuracy"] = global_acc

    # per-class accuracy
    class_acc = {}
    for i, cls in enumerate(class_names):
        idx = np.where(y_true == i)[0]
        if len(idx) > 0:
            class_acc[cls] = # YOUR CODE HERE...
        else:
            class_acc[cls] = None
    results["class_accuracy"] = class_acc

    # accuracy on seen classes
    seen_idx = [class_names.index(k) for k in seen_classes]
    mask = np.isin(y_true, seen_idx)

    if np.sum(mask) > 0:
        seen_acc = # YOUR CODE HERE...
    else:
        seen_acc = None
    results["seen_class_accuracy"] = seen_acc

    print(f"{center_name}")
    print(f"Global accuracy: {global_acc*100:.2f}%")
    print(f"Seen-classes accuracy: {seen_acc*100:.2f}%")

    print("\nClass-wise Accuracy:")
    for cls, acc in class_acc.items():
        print(f"{cls:<25} :  {acc*100:6.2f}%")

    print("\n")

    # plot a confusion matrix
    cm = # YOUR CODE HERE...

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=class_names)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    plt.title(f"{center_name} confusion matrix")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    return results


Separate out "missing" class to have stratified evaluation.

In [ ]:
ALL_CLASSES = CLASSES

seen_classes_c1 = [c for c in ALL_CLASSES if c != "Monocyte"]
seen_classes_c2 = [c for c in ALL_CLASSES if c != "Neutrophile Segment"]

How does Centre 1 perform?

In [ ]:
results_c1 = evaluate() # YOUR CODE HERE... don't forget to evaluate on the correct test set!

How does Centre 2 perform?

In [ ]:
results_c2 = evaluate() # YOUR CODE HERE... don't forget to evaluate on the correct test set!


## 4. Federated Learning

For now, we have seen:
- how each centre performs on its own
- how well or poorly each model performs on seen vs unseen classes

In this part, we will implement a federated learning (FL) algorithm to mitigate performance drops on unseen classes:

- We start with a **global model**

- Send the model to each centre

- Each centre trains locally on **its own data** (for `n_epochs_per_round`)

- Centres send back their **updated weights**

- We **average** these updates

- The improved global model is **redistributed**

- Repeat for `n_rounds`

Because the centres are missing different classes, sharing updates (weights) allows the global model to learn from both datasets.


<p align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*G7cu8M6UlMsssARt9Kp4iQ.jpeg"
       width="600">
</p>


Some helper functions to get and distribute weights:

In [ ]:
def get_weights(model):
    return model.get_weights()

def set_weights(model, weights):
    model.set_weights(weights)

Function to implement **federated averageing**
- Each centre’s weights are averaged and *weighted* by its number of training samples
=> Centres with more data have a proportionally *larger influence* on the global update

<br>**Hint:** Use formula above

In [ ]:
# compute federated averaging (weighted by nb of samples)
def weighted_fedavg(weights_list, sizes):
    total = sum(sizes)
    new_weights = []
    for layer_weights in zip(*weights_list):
        new_layer = np.zeros_like(layer_weights[0])
        for w, n in zip(layer_weights, sizes):

            new_layer += # YOUR CODE HERE... fill in new layer (see formula)

        new_weights.append(new_layer)
    return new_weights


Initialise **global**

In [ ]:
# size of centre 1 and centre 2 training sets
sizes = [train_c1.n, train_c2.n]

# one shared starting point for all clients
global_model = build_model()
global_weights = global_model.get_weights()
print("Global model initialised.")

**Start FL loop**

<br>**Hint:** Follow the steps described earlier (send global model --> local training --> return updates --> average --> repeat).

**Hint:** change number of rounds and epochs depending on your compute power. Ideally, you want at least 2 epochs per round and 2/3 rounds.

In [ ]:
num_rounds = # YOUR CODE HERE...
epochs_per_round = # YOUR CODE HERE...

# client models
model_c1_fl = # YOUR CODE HERE...
model_c2_fl = # YOUR CODE HERE...

for rnd in range(num_rounds):

    print(f"\n Round {rnd+1}")

    ## Client 1
    # Step 1: assign weights
    model_c1_fl.set_weights() # YOUR CODE HERE... which weights should you assign?
    # Step 2: train on updated weights (or initial weights if first loop)
    model_c1_fl.fit() # YOUR CODE HERE...
    # Step 3: save the new weights
    w1 = model_c1_fl. # YOUR CODE HERE...

    ## Client 2
    # YOUR CODE HERE... same as Client 1

    ## Federated averaging
    # Step 1: compute new averaged weights
    global_weights = # YOUR CODE HERE... hint: weighted_fedavg
    # Step 2: assign new weights to global model
    global_model. # YOUR CODE HERE...

    # global testing (just for monitoring)
    _, acc = global_model.evaluate(test_gen, verbose=0)
    print(f"Global accuracy after round {rnd+1}: {acc*100:.2f}%")


How does the Global model perform?
<br> How has class-wise performance changed?

In [ ]:
evaluate() # YOUR CODE HERE...

## 5. Federated learning with `flower`: you can also get familiar with an online package to run FL simulations

https://flower.ai/docs/framework/tutorial-quickstart-tensorflow.html

In [ ]:
!pip install -U "flwr[simulation]"
!pip install --upgrade pip setuptools wheel